# CLAMPS review bibliometric analysis — quick reproduction

This notebook rebuilds the **726-work review corpus** and regenerates the paper figures
from frozen pipeline checkpoints (~20 seconds on Binder).

| Layer | What |
|-------|------|
| **This notebook** | Interactive reproduction (Binder laboratory) |
| **`scripts/` + `PIPELINE.md`** | Full operational pipeline for your own facility |
| **Zenodo DOI** | Permanent archive for citation |

> **Note:** Binder runs the *quick path* only. Full OpenAlex discovery and PDF scanning
> require local setup — see `OPERATIONS.md` and `PIPELINE.md`.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Image, display, Markdown

ROOT = Path.cwd()
for candidate in (ROOT, ROOT.parent, ROOT.parent / "bibliometrics"):
    if (candidate / "scripts").is_dir() and (candidate / "checkpoints").is_dir():
        ROOT = candidate
        break

os.environ["MPLCONFIGDIR"] = str(ROOT / ".matplotlib")
(ROOT / ".matplotlib").mkdir(exist_ok=True)
(ROOT / "output" / "figures").mkdir(parents=True, exist_ok=True)
(ROOT / "output" / "review_metrics").mkdir(parents=True, exist_ok=True)

if not (ROOT / "config.yaml").exists():
    import shutil
    shutil.copy(ROOT / "config.yaml.example", ROOT / "config.yaml")

print(f"Project root: {ROOT}")

In [ ]:
# Restore checkpoints if output/ is empty (Binder postBuild usually does this)
CHECKPOINTS = [
    "clamps_papers_high_confidence_channels_with_pdfs.csv",
    "clamps_papers_discovered_channels.csv",
    "clamps_data_deposits.csv",
    "clamps_theses_master_list.csv",
    "clamps_publications_html_scan_log.csv",
    "clamps_publications_html_mentions.csv",
    "clamps_scan_log.csv",
    "clamps_text_mentions.csv",
]
for name in CHECKPOINTS:
    src = ROOT / "checkpoints" / name
    dst = ROOT / "output" / name
    if src.exists() and (not dst.exists() or dst.stat().st_size == 0):
        dst.write_bytes(src.read_bytes())

local_log = ROOT / "output" / "clamps_local_scan_log.csv"
if not local_log.exists() or local_log.stat().st_size == 0:
    local_log.write_text(
        "openalex_id,doi,status,mention_count,matched_terms,match_strength,error\n"
    )
print("Checkpoints ready.")

## Step 1 — Discovery funnel (simulated from checkpoints)

The full pipeline queries OpenAlex through **eight discovery channels** (A–H), then applies
filters before merging inclusion streams. This notebook does **not** re-query OpenAlex on Binder;
it replays the frozen checkpoint outputs so you can see how the 726-work corpus was assembled.

| Channel | Role |
|---------|------|
| **A** | Facility identity phrases (CLAMPS, Mobile PISA, …) |
| **B** | Campaign × instrument compound queries |
| **C** | Dataset DOI and grant citations |
| **D** | Author seed bibliographies |
| **E** | Campaign anchor-paper author groups |
| **F** | OpenAlex thesis institutions |
| **G** | Ground-truth DOI registry (mandatory bypass) |
| **H** | Institutional repository thesis crawl |

After channel merge: deduplication → topic exclusions → high-confidence gate → type filter.
The corpus then adds **validated datasets**, **ground-truth bypass works**, and **manual theses**
(four inclusion streams; overlaps removed at the end).

In [ ]:
sys.path.insert(0, str(ROOT))
from clamps_biblio.corpus_ledger import (
    CHANNEL_LABELS,
    discovery_channel_counts,
    funnel_summary_table,
    merge_stream_counts,
)

display(Markdown("### Discovery channels (checkpoint replay)"))
channels = discovery_channel_counts(ROOT)
if channels.empty:
    print("No discovery checkpoint found — run full pipeline locally.")
else:
    display(channels[["label", "works"]].rename(columns={"label": "channel", "works": "works_found"}))

display(Markdown("### Funnel steps"))
display(funnel_summary_table())

display(Markdown("### Inclusion streams → final corpus"))
display(merge_stream_counts())

### Merge streams into the 726-work corpus

In [ ]:
def run(script: str, *args: str) -> None:
    cmd = [sys.executable, str(ROOT / "scripts" / script), *args]
    print(f"\n>>> {' '.join(cmd)}")
    subprocess.run(cmd, check=True, cwd=ROOT)

run("ensure_mandatory_corpus_inputs.py")
run("build_review_corpus.py")
run("export_review_corpus_clean.py")

In [ ]:
corpus = pd.read_csv(ROOT / "output" / "clamps_review_corpus_clean.csv")
print(f"Corpus: {len(corpus)} works\n")
display(corpus["work_type"].value_counts().to_frame("count"))

if len(corpus) != 726:
    raise RuntimeError(f"Expected 726 works, got {len(corpus)}")

## Step 2 — Affiliation summary

NWC affiliation is inferred from OpenAlex co-author institutions. Works without resolvable
institution metadata are counted as **non-NWC** (no separate Unknown column).

In [ ]:
from clamps_biblio.review_metrics_data import load_corpus_review_frames
from clamps_biblio.nwc_affiliation import NWC_AFFILIATED, NON_AFFILIATED

frames = load_corpus_review_frames(ROOT)
all_works = frames["flagged_yd"]
pubs = frames["pdf_confirmed"]
peer = pubs[pubs["is_peer_reviewed"]]
theses = all_works[all_works["corpus_class"] == "thesis"]

def split(df, label):
    return {
        "metric": label,
        "total": len(df),
        "NWC": int((df["affiliation"] == NWC_AFFILIATED).sum()),
        "Non-NWC": int((df["affiliation"] == NON_AFFILIATED).sum()),
    }

summary = pd.DataFrame([
    split(all_works, "Full review corpus"),
    split(pubs, "Articles + reports"),
    split(peer, "Peer-reviewed (articles/reports)"),
    split(theses, "Theses"),
])
display(summary)

## Step 3 — Generate figures and tables

In [ ]:
run("plot_review_figures.py", "--no-archive")
print("Figures written to output/figures/")

In [ ]:
fig_dir = ROOT / "output" / "figures"
for name in [
    "fig_affiliation_by_year_with_deployments.png",
    "fig_work_type_by_year_with_deployments.png",
    "fig_campaign_by_year.png",
]:
    path = fig_dir / name
    if path.exists():
        display(Markdown(f"### `{name}`"))
        display(Image(filename=str(path)))

display(Markdown("### Corpus summary table"))
display(pd.read_csv(fig_dir / "table_corpus_summary.csv"))
display(Markdown("### Impact summary table"))
display(pd.read_csv(fig_dir / "table_impact_summary.csv"))

## Next steps

### Run the full pipeline on your own machine
Download this repository and follow **`OPERATIONS.md`** to adapt discovery channels,
ground-truth registries, and affiliation rules for another facility.

### Cite this work
Use the Zenodo DOI from **`CITATION.cff`** once published. See **`ZENODO.md`** for upload steps.

### Case data gallery (separate project)
Interactive CLAMPS observation case exploration will be hosted in the Case Gallery
repository with its own Binder notebook.